In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
torch.cuda.is_available()

True

In [6]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import math

In [5]:
def get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps: int,
        num_training_steps: int,
        num_cycles: float = 0.5,
        min_lr_ratio: float = 0.0,
):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return current_step / max(1, num_warmup_steps)
        
        progress = (current_step - num_warmup_steps) / max(1, num_training_steps - num_warmup_steps)
        progress = min(progress, 1.0)
        cosine_decay = 0.5 * (1.0 + math.cos(math.pi * num_cycles * 2.0 * progress))
        return max(min_lr_ratio, cosine_decay)

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

In [70]:
class ModelNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.ffn = nn.Linear(5, 1)

    def forward(self, X):
        return self.ffn(X)

class MockDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        x, y = self.data[idx]
        return x, y

def mock_collate_fn(batch):
    # batch is a list of (X, y) tuples, e.g.
    # [([1,2,3,4,5], [1]), ([2,3,4,5,6], [2]), ...]
    X_batch, y_batch = zip(*batch)  # unzips into two tuples of lists

    X_tensor = torch.tensor(X_batch, dtype=torch.float32)  # shape: (batch_size, 5)
    y_tensor = torch.tensor(y_batch, dtype=torch.float32)  # shape: (batch_size, 1)

    return X_tensor, y_tensor

In [ ]:
samples = [
    # X, y
    ([1, 2, 3, 4, 5], [1]),
    ([2, 3, 4, 5, 6], [2]),
    ([3, 4, 5, 6, 7], [3]),
    ([4, 5, 6, 7, 8], [4]),
    ([5, 6, 7, 8, 9], [5]),
]
dataset = MockDataset(data=samples)
train_loader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=mock_collate_fn)

In [91]:
x, y = zip(*(
    dataset[0],
    dataset[1]
))
x, y

(([1, 2, 3, 4, 5], [2, 3, 4, 5, 6]), ([1], [2]))

In [95]:
3e-4

0.0003

In [ ]:
model = ModelNetwork()
# follow GPT
betas = (0.9, 0.98) # b1, b2
# the Karpathy constant (Andrej Karpathy tweeted that 3e-4 is "the best learning rate for Adam, hands down"
lr = 3e-4
weight_decay = 0.1
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=lr, 
    betas=betas, 
    weight_decay=weight_decay
)

num_epochs = 100
num_training_steps = num_epochs * len(train_loader)
num_warmup_steps = int(0.05 * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

for epoch in range(num_epochs):
    model.train()
    for X, y_true in train_loader:
        optimizer.zero_grad()
        y_pred = model(X)
        loss = F.mse_loss(y_true, y_pred)
        loss.backward()
        optimizer.step()
        scheduler.step()

In [94]:
num_epochs, num_training_steps, num_warmup_steps

(100, 300, 15)